# Model Comparison and Error Analysis

In [ ]:
import numpy as np
import pandas as pd
import joblib

from pathlib import Path
from scipy.special import softmax

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)

from IPython.display import display

## Load baseline models

In [ ]:
baseline_data = joblib.load("baseline_models.pkl")

lr_model = baseline_data["lr_model"]
mlp = baseline_data["mlp"]
x_val = baseline_data["x_val"]
val_df = baseline_data["val_df"]

val_reset = val_df.reset_index(drop=True)
y_true = val_reset["sentiments"].values

print("Validation set:", val_reset.shape)

MODEL_DIR = "review_model/best_distilbert"

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

bert_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_DIR
)

bert_val_ds = Dataset.from_pandas(
    val_reset.rename(columns={"sentiments": "label"}),
    preserve_index=False
)

def preprocess_function(examples):
    return tokenizer(
        examples["reviews"],
        truncation=True
    )

tokenized_val = bert_val_ds.map(
    preprocess_function,
    batched=True
)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

training_args = TrainingArguments(
    output_dir="comparison_output",
    per_device_eval_batch_size=16,
    report_to="none"
)

bert_trainer = Trainer(
    model=bert_model,
    args=training_args,
    data_collator=data_collator
)

## Generate predictions

In [ ]:
# Logistic Regression and MLP
lr_probs = lr_model.predict_proba(x_val)
mlp_probs = mlp.predict_proba(x_val)

# DistilBERT
bert_logits = bert_trainer.predict(tokenized_val).predictions
bert_probs = softmax(bert_logits, axis=1)

## Create a comparison dataframe

In [ ]:
label_names = {
    0: "Negative",
    1: "Positive"
}

results = pd.DataFrame({
    "review": val_reset["reviews"].values,
    "true": y_true
})

for name, probs in [
    ("lr", lr_probs),
    ("mlp", mlp_probs),
    ("bert", bert_probs)
]:
    results[f"{name}_pred"] = probs.argmax(axis=1)
    results[f"{name}_conf"] = probs.max(axis=1)
    results[f"{name}_correct"] = (
        results[f"{name}_pred"] == results["true"]
    )

results.head()

In [ ]:
for name in ["lr", "mlp", "bert"]:

    correct = results[f"{name}_correct"]

    avg_conf_correct = results.loc[
        correct, f"{name}_conf"
    ].mean()

    avg_conf_wrong = results.loc[
        ~correct, f"{name}_conf"
    ].mean()

    print(
        f"{name.upper():5s} errors: {(~correct).sum()}/{len(results)} | "
        f"avg conf correct: {avg_conf_correct:.4f} | "
        f"avg conf wrong: {avg_conf_wrong:.4f}"
    )

## Individual error analysis

In [ ]:
def show_samples(name, n=2, seed=42):

    correct = results[results[f"{name}_correct"]]
    wrong = results[~results[f"{name}_correct"]]

    for title, subset in [
        ("CORRECT", correct),
        ("INCORRECT", wrong)
    ]:

        print(f"\n{'='*20} {name.upper()} - {title} {'='*20}")

        for i, row in subset.sample(
            min(n, len(subset)),
            random_state=seed
        ).iterrows():

            print(f"\nIndex: {i}")
            print(f"Review: {row['review'][:400]}")
            print(
                f"True: {label_names[row['true']]} | "
                f"Pred: {label_names[row[f'{name}_pred']]} | "
                f"Confidence: {row[f'{name}_conf']:.4f}"
            )

In [ ]:
for model_name in ["lr", "mlp", "bert"]:
    show_samples(model_name)

In [ ]:
def compare_row(idx):

    r = results.loc[idx]

    print(f"\nIndex {idx} | True: {label_names[r['true']]}")
    print(f"Review: {r['review'][:400]}")

    for m, nm in [
        ("lr", "LogReg"),
        ("mlp", "MLP"),
        ("bert", "DistilBERT")
    ]:

        mark = "✓" if r[f"{m}_correct"] else "✗"

        print(
            f"  {nm:10s} {mark} "
            f"Pred: {label_names[r[f'{m}_pred']]:8s} "
            f"conf: {r[f'{m}_conf']:.4f}"
        )

In [ ]:
# (a) A review all three models classified correctly

all_right = results[
    results.lr_correct &
    results.mlp_correct &
    results.bert_correct
]

print("##### ALL THREE CORRECT (strength) #####")

for idx in all_right.sample(
    min(1, len(all_right)),
    random_state=42
).index:
    compare_row(idx)

In [ ]:
# (b) Reviews where DistilBERT is correct but both baselines are wrong

bert_only = results[
    results.bert_correct &
    ~results.lr_correct &
    ~results.mlp_correct
]

print("\n##### ONLY DISTILBERT CORRECT #####")

for idx in bert_only.sample(
    min(2, len(bert_only)),
    random_state=42
).index:
    compare_row(idx)

In [ ]:
# (c) Reviews where DistilBERT is wrong

bert_wrong = results[
    ~results.bert_correct
]

print("\n##### DISTILBERT WRONG (weakness) #####")

for idx in bert_wrong.sample(
    min(2, len(bert_wrong)),
    random_state=42
).index:
    compare_row(idx)

In [ ]:
# (d) Most confident DistilBERT mistakes

print("\n##### MOST CONFIDENT DISTILBERT ERRORS #####")

for idx in bert_wrong.sort_values(
    "bert_conf",
    ascending=False
).head(2).index:
    compare_row(idx)

In [ ]:
report_samples = {}

# One example where all three models are correct
if len(all_right) > 0:
    report_samples[all_right.sample(1, random_state=42).index[0]] = "All correct"

# Up to two examples where only DistilBERT is correct
for idx in bert_only.sample(
    min(2, len(bert_only)),
    random_state=42
).index:
    report_samples[idx] = "Only DistilBERT correct"

# Up to two examples where DistilBERT is wrong
for idx in bert_wrong.sample(
    min(2, len(bert_wrong)),
    random_state=42
).index:
    report_samples[idx] = "DistilBERT wrong"

# Up to two examples of confident DistilBERT errors
for idx in bert_wrong.sort_values(
    "bert_conf",
    ascending=False
).head(2).index:
    report_samples[idx] = "Confident DistilBERT error"

report_idx = list(report_samples.keys())

print("Selected review indices:", report_idx)

In [ ]:
def fmt(r, m):

    mark = "✓" if r[f"{m}_correct"] else "✗"

    return (
        f"{label_names[r[f'{m}_pred']]} "
        f"{mark} ({r[f'{m}_conf']:.2f})"
    )


rows = []

for i in report_idx:

    r = results.loc[i]

    rows.append({
        "Idx": i,
        "Review (excerpt)": r["review"][:120] + "...",
        "True": label_names[r["true"]],
        "LogReg": fmt(r, "lr"),
        "MLP": fmt(r, "mlp"),
        "DistilBERT": fmt(r, "bert"),
        "Category": report_samples[i]
    })

report_table = pd.DataFrame(rows)

display(report_table)

In [ ]:
report_table.to_csv(
    "report_table.csv",
    index=False
)

print(report_table.to_markdown(index=False))

## Generate LaTeX tables

In [ ]:
def esc(s):

    s = " ".join(str(s).split())

    s = s.replace("\\", "@@BS@@")

    for ch in "&%$#_{}":
        s = s.replace(ch, "\\" + ch)

    return (
        s.replace("~", r"\textasciitilde{}")
         .replace("^", r"\textasciicircum{}")
         .replace('"', "''")
         .replace("@@BS@@", r"\textbackslash{}")
    )


def category(r):

    lr = r.lr_correct
    mlp = r.mlp_correct
    bert = r.bert_correct

    if lr and mlp and bert:
        return "All correct"

    if bert and not lr and not mlp:
        return "Only DistilBERT correct"

    if not (lr or mlp or bert):
        return "All wrong"

    if not bert:
        return "DistilBERT wrong"

    return "Mixed"


def cell(r, m):

    mark = r"$\checkmark$" if r[f"{m}_correct"] else r"$\times$"

    return (
        f"{label_names[r[f'{m}_pred']][:3]} "
        f"{mark} ({r[f'{m}_conf']:.2f})"
    )

In [ ]:
rows1 = []

for i in report_idx:

    r = results.loc[i]

    rows1.append(
        " & ".join([
            str(i),
            label_names[r["true"]][:3],
            cell(r, "lr"),
            cell(r, "mlp"),
            cell(r, "bert"),
            category(r)
        ]) + r" \\"
    )


table1 = r"""\begin{table}[htbp]
\centering
\small
\begin{tabular}{r l l l l l}
\toprule
\textbf{Idx} & \textbf{True} & \textbf{LogReg} & \textbf{MLP} & \textbf{DistilBERT} & \textbf{Category} \\
\midrule
""" + "\n".join(rows1) + r"""
\bottomrule
\end{tabular}
\caption{Predicted label, correctness and confidence for selected validation reviews.}
\label{tab:samples}
\end{table}"""

print(table1)

In [ ]:
rows2 = [
    f"{i} & {esc(results.loc[i, 'review'])} \\\\"
    for i in report_idx
]

table2 = r"""\begin{table}[htbp]
\centering
\small
\renewcommand{\arraystretch}{1.3}
\begin{tabularx}{\textwidth}{r X}
\toprule
\textbf{Idx} & \textbf{Review text} \\
\midrule
""" + "\n".join(rows2) + r"""
\bottomrule
\end{tabularx}
\caption{Full text of the reviews referenced in Table~\ref{tab:samples}.}
\label{tab:sample-text}
\end{table}"""

print(table2)

In [ ]:
results.to_csv(
    "all_models_validation_predictions.csv",
    index=False
)

print("All model predictions saved successfully.")